##Loading Datasets From Hugging Face

In [ ]:
!pip install transformers torch datasets

In [ ]:
from datasets import load_dataset
dataset = load_dataset("fka/awesome-chatgpt-prompts")
dataset

In [ ]:
!pip install py7zr

In [ ]:
from datasets import load_dataset

dataset = load_dataset("knkarthick/samsum")
dataset

### Data Preprocessing Methods

In [ ]:
dataset = load_dataset("fka/awesome-chatgpt-prompts")
dataset

In [ ]:
dataset['train'][0]

In [ ]:
# shuffle & sample

dataset = dataset['train'].shuffle(seed=37).select(range(100))
dataset

In [ ]:
# Create Test Dataset

dataset = dataset.train_test_split(train_size=0.8, seed=42)
dataset

###Creating custom Own Dataset

In [ ]:
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/reuters21578-mld/reuters21578.tar.gz

In [ ]:
!tar -xzvf reuters21578.tar.gz

In [ ]:
# the sgm files are what contains the articles
from bs4 import BeautifulSoup

# Open the file and parse its content with BeautifulSoup
reuters_articles = []
for i in range(22):
  if i < 10:
    i = f"0{i}"

  # load file data
  with open(f"/content/reut2-0{i}.sgm", 'r', encoding='latin-1') as file:
      soup = BeautifulSoup(file, "html.parser")

  # Extract articles titles and bodies
  articles = []
  for reuters in soup.find_all('reuters'):
      title = reuters.title.string if reuters.title else ""
      body = reuters.body.string if reuters.body else ""
      articles.append({
            'title': title,
            'body': body
        })

  reuters_articles.extend(articles)

In [ ]:
# Print first few articles for inspection
for i, article in enumerate(reuters_articles[:5]):
  print(article)
  print("-"*100)
print(f" articles len ==> {len(reuters_articles)}")


In [ ]:
import json

TRAIN_PCT, VALID_PCT = 0.8, 0.1
# Split the data
train_articles = reuters_articles[:int(len(reuters_articles)*TRAIN_PCT)]
valid_articles = reuters_articles[int(len(reuters_articles)*TRAIN_PCT): int(len(reuters_articles)*(TRAIN_PCT + VALID_PCT))]
test_articles = reuters_articles[int(len(reuters_articles)*(TRAIN_PCT + VALID_PCT)):]

# Function to save articles as JSONL
def save_as_jsonl(data, filename):
    with open(filename, "w") as f:
        for article in data:
            f.write(json.dumps(article) + "\n")

# Save them into temporary JSONL files
save_as_jsonl(train_articles, "train.jsonl")
save_as_jsonl(valid_articles, "valid.jsonl")
save_as_jsonl(test_articles, "test.jsonl")

In [ ]:
# Load them as datasets
data_files = {"train": "train.jsonl", "validation": "valid.jsonl", "test": "test.jsonl"}
dataset = load_dataset("json", data_files=data_files)
dataset

In [ ]:
dataset['train'][0]

### Upload Dataset to hugging face Hub

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
dataset

In [ ]:
dataset.push_to_hub("custom_reuters_articles")

##Custom Tokenizer setup

In [ ]:
!pip install transformers datasets torch

In [ ]:
from datasets import load_dataset
dataset = load_dataset("HFS26/custom_reuters_articles")
dataset

### Creating New Column

In [ ]:
def create_full_article_col(example):

  return {'full_article': f"TITLE:{example['title']}\n\nBODY:{example['body']}"}

dataset = dataset.map(create_full_article_col)
dataset

In [ ]:
dataset['train'][0]['full_article']

### Training own Tokenizer

In [ ]:
# Create a batched dataset for training, creates an iterator object for later usage when training tokenizer

training_corpus = (
    dataset["train"][i : i + 1000]["full_article"]
    for i in range(0, len(dataset["train"]), 1000)
)

In [ ]:
#Here we are fine-tuning a pre-trained tokenizer instead of creating one from scratch. This makes training a lot easier and faster.
from transformers import AutoTokenizer
prev_tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer = prev_tokenizer.train_new_from_iterator(training_corpus, 52000) # vocab size of 52000

In [ ]:
example = dataset['test'][2]['full_article']
example

In [ ]:
prev_tokenizer.tokenize(example)

In [ ]:
tokenizer.tokenize(example)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
tokenizer.push_to_hub("gpt2-reuters-tokenizer")

### Using Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HFS26/gpt2-reuters-tokenizer")

In [ ]:
example = dataset['test'][2]
example

In [ ]:
tokenizer.tokenize(example['full_article'])